In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from figure_formatting import setup_figure, save_figure
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [2]:
import logging
import matplotlib as mpl

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]

In [3]:
PREFIXES = ['Association_', 'Cerebellum_', 'Commissure_', 'ProjectionBrainstem_', 'ProjectionBasalGanglia_']
STOPWORDS = {"of", "and", "in", "for", "to", "with", "on", "at"}

# regexes used repeatedly
RE_CAMEL_SPACE = re.compile(r'(?<!^)(?=[A-Z])') # capital letter that is not the first character in the string
RE_LR_END = re.compile(r'(L|R)$') # l/r at end of string
RE_MSMT_PREFIX = re.compile(r'^bundle_') # msmt prefix
RE_MM_SUFFIX = re.compile(r'\s*mm[23]?\s*$', flags=re.IGNORECASE) # trailing millimeter units e..g mm, mm2, mm3

def clean_metrics_name(name: str) -> str:
    name = str(name).replace('_', ' ')
    name = RE_MM_SUFFIX.sub('', name).strip() # remove mm units

    upper = name.upper()
    if upper.startswith("NODDI") or upper.startswith("DKI"):
        return " ".join(w.upper() for w in name.split())

    ordinal_allowed = "quarter" in name.lower()

    def ordinalize(token: str) -> str:
        n = int(token)
        if n % 10 == 1 and n % 100 != 11: return f"{n}st"
        if n % 10 == 2 and n % 100 != 12: return f"{n}nd"
        if n % 10 == 3 and n % 100 != 13: return f"{n}rd"
        return f"{n}th"

    cleaned = []
    for w in name.split():
        if w.isdigit():
            cleaned.append(ordinalize(w) if ordinal_allowed else w)
        elif w.lower() in STOPWORDS:
            cleaned.append(w.lower())
        else:
            cleaned.append(w.capitalize())
    return " ".join(cleaned)


In [4]:
# ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
INPUT_DIR = ROOT_DIR / "input_data"

abbrevs = pd.read_excel(INPUT_DIR / "abbreviations.xlsx", sheet_name="Sheet1")
annots = pd.read_csv(INPUT_DIR / "tract_sa_axis_ranges_thresh50.csv")

tracts = ['AssociationArcuateFasciculusL','AssociationArcuateFasciculusR', 'AssociationCingulumL', 'AssociationCingulumR',
          'AssociationExtremeCapsuleL', 'AssociationExtremeCapsuleR', 'AssociationFrontalAslantTractL','AssociationFrontalAslantTractR',
          'AssociationHippocampusAlveusL', 'AssociationHippocampusAlveusR', 'AssociationInferiorFrontoOccipitalFasciculusL',
          'AssociationInferiorFrontoOccipitalFasciculusR','AssociationInferiorLongitudinalFasciculusL',
          'AssociationInferiorLongitudinalFasciculusR','AssociationMiddleLongitudinalFasciculusL', 'AssociationMiddleLongitudinalFasciculusR',
          'AssociationParietalAslantTractL','AssociationParietalAslantTractR','AssociationSuperiorLongitudinalFasciculusL',
          'AssociationSuperiorLongitudinalFasciculusR', 'AssociationUncinateFasciculusL', 'AssociationUncinateFasciculusR',
          'AssociationVerticalOccipitalFasciculusL', 'AssociationVerticalOccipitalFasciculusR','CerebellumCerebellumL',
          'CerebellumCerebellumR','CerebellumInferiorCerebellarPeduncleL','CerebellumInferiorCerebellarPeduncleR',
          'CerebellumMiddleCerebellarPeduncle', 'CerebellumSuperiorCerebellarPeduncle','CerebellumVermis', 'CommissureCorpusCallosum',
          'ProjectionBasalGangliaAcousticRadiationL', 'ProjectionBasalGangliaAcousticRadiationR','ProjectionBasalGangliaAnsaLenticularisL',
          'ProjectionBasalGangliaAnsaLenticularisR', 'ProjectionBasalGangliaAnsaSubthalamicaL', 'ProjectionBasalGangliaAnsaSubthalamicaR',
          'ProjectionBasalGangliaCorticostriatalTractL', 'ProjectionBasalGangliaCorticostriatalTractR', 
          'ProjectionBasalGangliaFasciculusLenticularisL','ProjectionBasalGangliaFasciculusLenticularisR',
          'ProjectionBasalGangliaFasciculusSubthalamicusL', 'ProjectionBasalGangliaFasciculusSubthalamicusR', 'ProjectionBasalGangliaFornixL',
          'ProjectionBasalGangliaFornixR','ProjectionBasalGangliaOpticRadiationL', 'ProjectionBasalGangliaOpticRadiationR',
          'ProjectionBasalGangliaThalamicRadiationL', 'ProjectionBasalGangliaThalamicRadiationR', 'ProjectionBrainstemCorticopontineTractL',
          'ProjectionBrainstemCorticopontineTractR', 'ProjectionBrainstemCorticospinalTractL', 'ProjectionBrainstemCorticospinalTractR',
          'ProjectionBrainstemMedialForebrainBundleL', 'ProjectionBrainstemMedialForebrainBundleR', 'ProjectionBrainstemMedialLemniscusL',
          'ProjectionBrainstemMedialLemniscusR', 'ProjectionBrainstemNonDecussatingDentatorubrothalamicTractL',
          'ProjectionBrainstemNonDecussatingDentatorubrothalamicTractR', 'ProjectionBrainstemReticularTractL', 
          'ProjectionBrainstemReticularTractR']

prefixes = ["Association", "Cerebellum", "Commissure", "Projection"]

def add_prefix_underscore(tract):
    for p in prefixes:
        if tract.startswith(p):
            return tract.replace(p, p + "_", 1)
    return tract

tracts = [add_prefix_underscore(t) for t in tracts]

In [5]:
abbrevs.head(2)

,Tract,Abbreviation,HCPYA_1065,Hemisphere,Full_Name,Full_Name_capitalized,Tract_Long_Name,new_qsirecon_tract_names,Other_Nomenclatures,Type
0,AF_left,AF,AF_L,Left,arcuate fasciculus,Left Arcuate Fasciculus,Arcuate_Fasciculus_L,Association_ArcuateFasciculusL,NaN,Association
1,C_FPH_left,C_FPH,C_FPH_L,Left,"cingulum, frontal parahippocampal segment","Left Cingulum, Frontal Parahippocampal Segment",Cingulum_Frontal_Parahippocampal_L,NaN,NaN,Association


In [6]:
annots.head(2)

,Tract,SA_Range,N_Regions,Hemisphere,Abbreviation
0,FAT_left,168.0,26,left,FAT
1,AF_left,173.0,46,left,AF


In [7]:
abbrev_lookup = abbrevs[["new_qsirecon_tract_names", "Tract"]].drop_duplicates()
annot_lookup = annots[["Tract", "SA_Range"]].drop_duplicates()

abbrev_list = abbrev_lookup["new_qsirecon_tract_names"].dropna().tolist()
missing_tracts = [t for t in tracts if t not in abbrev_list]

print("Tracts not found in abbrevs['new_qsirecon_tract_names']:")
print(missing_tracts)
print(f"Total missing: {len(missing_tracts)}\n")

tracts_df = pd.DataFrame({"new_qsirecon_tract_names": tracts})

tracts_with_sa = (
    tracts_df
    .merge(abbrev_lookup, on="new_qsirecon_tract_names", how="left")
    .merge(annot_lookup, on="Tract", how="left")
)

tracts_with_sa_clean = tracts_with_sa.dropna()

for _, row in tracts_with_sa_clean.iterrows():
    print(row.to_dict())

Tracts not found in abbrevs['new_qsirecon_tract_names']:
['Association_ExtremeCapsuleL', 'Association_ExtremeCapsuleR', 'Association_HippocampusAlveusL', 'Association_HippocampusAlveusR', 'Cerebellum_CerebellumL', 'Cerebellum_CerebellumR', 'Cerebellum_InferiorCerebellarPeduncleL', 'Cerebellum_InferiorCerebellarPeduncleR', 'Cerebellum_MiddleCerebellarPeduncle', 'Cerebellum_SuperiorCerebellarPeduncle', 'Cerebellum_Vermis', 'Commissure_CorpusCallosum', 'Projection_BasalGangliaAcousticRadiationL', 'Projection_BasalGangliaAcousticRadiationR', 'Projection_BasalGangliaAnsaLenticularisL', 'Projection_BasalGangliaAnsaLenticularisR', 'Projection_BasalGangliaAnsaSubthalamicaL', 'Projection_BasalGangliaAnsaSubthalamicaR', 'Projection_BasalGangliaCorticostriatalTractL', 'Projection_BasalGangliaCorticostriatalTractR', 'Projection_BasalGangliaFasciculusLenticularisL', 'Projection_BasalGangliaFasciculusLenticularisR', 'Projection_BasalGangliaFasciculusSubthalamicusL', 'Projection_BasalGangliaFasciculu

In [8]:
tracts_with_sa_clean

,new_qsirecon_tract_names,Tract,SA_Range
0,Association_ArcuateFasciculusL,AF_left,173.0
1,Association_ArcuateFasciculusR,AF_right,140.0
2,Association_CingulumL,C_FP_left,116.0
3,Association_CingulumR,C_FP_right,159.0
6,Association_FrontalAslantTractL,FAT_left,168.0
7,Association_FrontalAslantTractR,FAT_right,156.0
10,Association_InferiorFrontoOccipitalFasciculusL,IFOF_left,171.0
11,Association_InferiorFrontoOccipitalFasciculusR,IFOF_right,171.0
12,Association_InferiorLongitudinalFasciculusL,ILF_left,167.0
13,Association_InferiorLongitudinalFasciculusR,ILF_right,178.0


In [9]:
FIG3_DIR = ROOT_DIR / "output_data" / "results" / "main_figures" / "figure3"
SUPPFIG4_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig4"
for d in [FIG3_DIR, SUPPFIG4_DIR]:
    d.mkdir(parents=True, exist_ok=True)

scenarios = [("NODDI", False), ("DKI", False)]
split_modes = ["avg", "A", "B"]

for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    for split_mode in split_modes:
        print(f"  [SPLIT_MODE] {split_mode}")

        if scenario_name == "NODDI" and scenario_etiv is False:
            save_dir = FIG3_DIR
            out_base = f"pca_ALL_LRkept_{split_mode}"
            file_stub = f"PC1_loading_and_scatter_{split_mode}"
        elif scenario_name == "DKI" and scenario_etiv is False:
            save_dir = SUPPFIG4_DIR
            out_base = f"pca_ALL_LRkept_dki_{split_mode}"
            file_stub = f"PC1_loading_and_scatter_dki_{split_mode}"
        else:
            continue

        pc_scores_df = pd.read_csv(save_dir / f"{out_base}_bundle_scores_PC12.csv")
        pc_loadings_df = pd.read_csv(save_dir / f"{out_base}_metric_loadings_PC12.csv")

        pc1_loadings = pc_loadings_df.set_index("metric")["PC1"].dropna().sort_values(ascending=True)
        pc1_loadings.index = [clean_metrics_name(m) for m in pc1_loadings.index]

        if pc1_loadings.index.duplicated().any():
            counts = {}
            new_idx = []
            for lbl in pc1_loadings.index:
                counts[lbl] = counts.get(lbl, 0) + 1
                new_idx.append(lbl if counts[lbl] == 1 else f"{lbl} ({counts[lbl]})")
            pc1_loadings.index = new_idx

        df_merged = pc_scores_df.merge(
            tracts_with_sa_clean[["new_qsirecon_tract_names", "SA_Range"]],
            left_on="bundle",
            right_on="new_qsirecon_tract_names",
            how="inner"
        )
        data_scatter = df_merged[["SA_Range", "PC1"]].dropna()

        x = data_scatter["SA_Range"].values
        y = data_scatter["PC1"].values
        r, _ = pearsonr(x, y)

        n_perm = 10000
        r_perm = np.empty(n_perm)
        rng = np.random.default_rng(0)
        for i in range(n_perm):
            x_perm = rng.permutation(x)
            r_perm[i], _ = pearsonr(x_perm, y)

        p_perm = (np.sum(np.abs(r_perm) >= np.abs(r)) + 1) / (n_perm + 1)

        if scenario_name == "DKI":
            height_mm = 90 
        else:
            height_mm = 75

        sns.set_style("white")
        fig, axes = setup_figure(
            width_mm=150,
            height_mm=height_mm,
            margins_mm=(42, 10, 10, 2),
            axes_linewidth=0.8,
            nrows=1,
            ncols=2,
            sharex=False,
            sharey=False
        )
        ax0, ax1 = axes.ravel()
        fig.subplots_adjust(wspace=0.35)

        ax0.barh(pc1_loadings.index, pc1_loadings.values, color="#7BA6E6")
        ax0.axvline(0, color="black", lw=1)
        ax0.set_xlabel("PC1 Loadings")
        ax0.set_ylabel("")
        sns.despine(ax=ax0, top=True, right=True)

        sns.regplot(
            data=data_scatter,
            x="SA_Range",
            y="PC1",
            scatter=False,
            ci=95,
            line_kws={"color": "black", "lw": 2},
            ax=ax1
        )
        sc = ax1.scatter(
            data_scatter["SA_Range"],
            data_scatter["PC1"],
            c=data_scatter["PC1"],
            cmap="viridis",
            s=45,
            alpha=0.85,
            edgecolor="none"
        )

        ax1.set_xlabel("SA Range")
        ax1.set_ylabel("PC1 Score")
        sns.despine(ax=ax1, top=True, right=True)

        ax1.text(
            0.05, 0.92,
            rf"$\it{{r}}$ = {r:.2f}" + "\n" + rf"$\it{{p}}_{{perm}}$ = {p_perm:.3f}",
            transform=ax1.transAxes,
            fontsize=7,
            ha="left",
            va="top"
        )

        ax0.tick_params(axis="both", which="both", bottom=True, left=True)
        ax1.tick_params(axis="both", which="both", bottom=True, left=True)

        cax = inset_axes(
            ax1, width="3%", height="30%", loc="upper left",
            bbox_to_anchor=(1.02, 0, 1, 1),
            bbox_transform=ax1.transAxes,
            borderpad=0
        )
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label("PC1 score")
        cbar.set_ticks([-6, 0, 6])
        cbar.set_ticklabels(["-6", "0", "6"])
        cbar.ax.tick_params(which="both", length=0)

        svg_path = save_dir / f"{file_stub}.svg"
        png_path = save_dir / f"{file_stub}.png"
        save_figure(fig, svg_path, autofit=False)
        fig.savefig(png_path, dpi=600, bbox_inches="tight")

        print(f"[SAVE] → {svg_path}")
        print(f"[SAVE] → {png_path}")
        plt.close(fig)


[SCENARIO] NODDI | eTIV=False
  [SPLIT_MODE] avg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_avg.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_avg.png
  [SPLIT_MODE] A
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_A.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_A.png
  [SPLIT_MODE] B
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_B.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/PC1_loading_and_scatter_B.png

[SCEN

In [10]:
from scipy.stats import pearsonr

scenarios = [("NODDI", False), ("DKI", False)]

for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    if scenario_name == "NODDI" and scenario_etiv is False:
        save_dir = FIG3_DIR
        out_base_A = "pca_ALL_LRkept_A"
        out_base_B = "pca_ALL_LRkept_B"
        out_stub = "loading_corr_splitA_vs_splitB"
    elif scenario_name == "DKI" and scenario_etiv is False:
        save_dir = SUPPFIG4_DIR
        out_base_A = "pca_ALL_LRkept_dki_A"
        out_base_B = "pca_ALL_LRkept_dki_B"
        out_stub = "loading_corr_splitA_vs_splitB_dki"
    else:
        continue

    # read saved loading tables
    loadings_A = pd.read_csv(save_dir / f"{out_base_A}_metric_loadings_PC12.csv")
    loadings_B = pd.read_csv(save_dir / f"{out_base_B}_metric_loadings_PC12.csv")

    # merge on metric
    merged = loadings_A.merge(
        loadings_B,
        on="metric",
        suffixes=("_A", "_B"),
        how="inner"
    )

    results = []

    for pc in ["PC1", "PC2"]:
        x = merged[f"{pc}_A"].values
        y = merged[f"{pc}_B"].values

        # sign-align split B to split A so PCA sign flips do not look unstable
        r_raw, _ = pearsonr(x, y)
        if r_raw < 0:
            merged[f"{pc}_B"] = -merged[f"{pc}_B"]
            y = merged[f"{pc}_B"].values

        r, p = pearsonr(x, y)

        results.append({
            "PC": pc,
            "r": r,
            "p": p,
            "n_metrics": len(x)
        })

        print(f"{pc}: r = {r:.3f}, p = {p:.4g}, n = {len(x)}")

    results_df = pd.DataFrame(results)
    results_path = save_dir / f"{out_stub}_PC12.csv"
    results_df.to_csv(results_path, index=False)

    print(f"[SAVE] → {results_path}")


[SCENARIO] NODDI | eTIV=False
PC1: r = 0.973, p = 5.606e-13, n = 20
PC2: r = 0.990, p = 1.25e-16, n = 20
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/loading_corr_splitA_vs_splitB_PC12.csv

[SCENARIO] DKI | eTIV=False
PC1: r = 0.955, p = 3.701e-14, n = 26
PC2: r = 0.982, p = 7.228e-19, n = 26
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig4/loading_corr_splitA_vs_splitB_dki_PC12.csv
